# 05 - TF-IDF Feature Engineering

Build explainable TF-IDF features from name + subtypes_clean + type. Category is intentionally excluded.


In [ ]:
import pickle
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "merged_dataset.csv")

required_columns = {"name", "subtypes", "type", "price_estimate"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Merged dataset is missing required columns: {sorted(missing)}")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_subtypes(text):
    text = clean_text(text)
    remove_words = {
        "attraction", "attractions", "place", "point", "location",
        "business", "service", "services", "establishment", "tourist"
    }
    words = [word for word in text.split() if word not in remove_words]
    return " ".join(words) if words else "unknown"


def enrich_keywords(text):
    text = str(text).lower()
    enrichment_map = {
        "pantai": ["beach"],
        "beach": ["pantai"],
        "gunung": ["mountain"],
        "air terjun": ["waterfall"],
        "danau": ["lake"],
        "gua": ["cave"],
        "taman": ["park"],
        "candi": ["temple"],
        "temple": ["candi"],
        "museum": ["museum"],
        "warung": ["food", "kuliner"],
        "cafe": ["coffee", "kuliner"],
        "coffee": ["cafe", "kopi"],
        "kopi": ["coffee", "cafe"],
        "restoran": ["restaurant", "kuliner"],
        "restaurant": ["restoran", "kuliner"],
        "hotel": ["lodging", "penginapan"],
        "penginapan": ["hotel", "lodging"],
        "resort": ["hotel", "lodging"],
        "villa": ["hotel", "lodging"],
        "murah": ["budget", "cheap"],
        "budget": ["murah", "cheap"],
    }

    extra_terms = []
    for keyword, terms in enrichment_map.items():
        if re.search(rf"\b{re.escape(keyword)}\b", text):
            extra_terms.extend(terms)
    return " ".join([text] + extra_terms)


df["subtypes"] = df["subtypes"].fillna("unknown").replace("", "unknown")
df["subtypes_clean"] = df["subtypes"].apply(clean_subtypes)

# Academic feature contract: name + subtypes_clean + type. Category is not used.
df["text"] = (
    df["name"].fillna("") + " " +
    df["subtypes_clean"].fillna("unknown") + " " +
    df["type"].fillna("")
)
df["text"] = df["text"].apply(enrich_keywords).apply(clean_text)

if df["text"].str.len().eq(0).any():
    raise AssertionError("Empty TF-IDF text found.")
if df["subtypes_clean"].isna().any() or df["subtypes_clean"].eq("").any():
    raise AssertionError("subtypes_clean must not be null or empty.")

df[["name", "type", "subtypes", "subtypes_clean", "text"]].head()


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
)

# Fit exactly once on the final engineered text.
tfidf_matrix = vectorizer.fit_transform(df["text"])

print("TF-IDF shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print(vectorizer.get_feature_names_out()[:30])


In [ ]:
if tfidf_matrix.shape[0] != len(df):
    raise AssertionError("TF-IDF row count does not match dataset row count.")
if vectorizer.ngram_range != (1, 2):
    raise AssertionError("TF-IDF must use ngram_range=(1, 2).")
if "category" in ["name", "subtypes_clean", "type"]:
    raise AssertionError("category must not be used in TF-IDF text.")

with open(DATA_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
with open(DATA_DIR / "tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

df.to_csv(DATA_DIR / "tfidf_dataset.csv", index=False)
print("Saved vectorizer, matrix, and TF-IDF-ready dataset.")
